# Contrastive RL (Acme-free port) — Colab runner

Binary **NCE** (contrastive_nce) on continuous-control goal-conditioned tasks.
All logic lives in the `crl/` package; this notebook only wires up Colab.

**How to use:** run cells 1-4 once, set `ENV_NAME` / `SEED` in **cell 5**, then
run cells 6-8. Checkpoints + metrics save to Google Drive on every eval.

No dataset needed (online RL). Time on a T4: `fetch_reach` ~20-40 min (300k),
`fetch_push` ~1 h (500k). Watch the printed `sps=`; ETA = steps / sps / 60 min.

In [ ]:
# 1. Install dependencies (~2 min).
!pip -q install "jax[cuda12]" dm-haiku optax gymnasium gymnasium-robotics mujoco imageio
import os
os.environ["MUJOCO_GL"] = "egl"  # headless MuJoCo rendering backend
import jax; print("JAX devices:", jax.devices())

In [ ]:
# 2. Get / update the code (your fork).
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
!git pull

In [ ]:
# 3. Mount Google Drive (checkpoints + reports are written here).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. ===== SET THESE, then run the rest =====
ENV_NAME = "fetch_push"      # "fetch_reach" or "fetch_push"
SEED     = 0                 # 0, 1, 2, ... (one run per seed)
STEPS    = 500_000           # fetch_reach: 300_000 ;  fetch_push: 500_000+
RESUME   = False             # True to continue a previous run of same ENV/SEED
# ==========================================
RUN_DIR = f"/content/drive/MyDrive/contrastive_rl_runs/{ENV_NAME}_nce_s{SEED}"
REPORT_DIR = f"/content/drive/MyDrive/repro_report_{ENV_NAME}_nce"
print("ENV_NAME:", ENV_NAME, "| SEED:", SEED, "| STEPS:", STEPS, "| RESUME:", RESUME)
print("RUN_DIR :", RUN_DIR)

In [ ]:
# 5. Train the binary-NCE baseline (guard-railed: refuses non-NCE configs).
from crl.config import Config
from crl.train import train

config = Config(
    env_name=ENV_NAME,
    use_td=False, twin_q=False,       # binary NCE (contrastive_nce)
    entropy_coefficient=0.0,
    max_number_of_steps=STEPS,
    min_replay_size=10_000, random_steps=10_000,
    num_sgd_steps_per_step=4,
    eval_every_steps=10_000, eval_episodes=20,
    seed=SEED,
    ckpt_dir=RUN_DIR,
    resume=RESUME,
)
assert config.use_td is False and config.twin_q is False, "must be binary NCE"
print(f"OK: {ENV_NAME}, binary NCE, seed {SEED} -> {RUN_DIR}")
state = train(config)

In [ ]:
# 6. Build the reproduction report (plots + GIFs + csv + audit) for this run.
!python -m crl.repro_report --env_name {ENV_NAME} --batch_size 256 --run_dirs {RUN_DIR} --out {REPORT_DIR}

In [ ]:
# 7. Show the report figures + rollout GIFs inline.
import os
from IPython.display import Image, display
for f in ["learning_curves.png", "distance_curves.png", "contrastive_logits.png",
          "ranking_accuracy.png", "rollout_random.gif", "rollout_trained.gif"]:
    p = os.path.join(REPORT_DIR, f)
    if os.path.exists(p):
        print(f); display(Image(p))

## Multi-seed report (optional)
For mean±std, run cells 4-6 with `SEED = 0`, then `1`, then `2` (three separate
runs), then run the cell below to combine them into one report.

In [ ]:
# 8. (optional) Combine seeds 0/1/2 into one mean+-std report.
ENV = "fetch_reach"   # or "fetch_push"
base = "/content/drive/MyDrive/contrastive_rl_runs"
d0, d1, d2 = (f"{base}/{ENV}_nce_s0", f"{base}/{ENV}_nce_s1", f"{base}/{ENV}_nce_s2")
out = f"/content/drive/MyDrive/repro_report_{ENV}_nce_3seed"
!python -m crl.repro_report --env_name {ENV} --batch_size 256 --run_dirs {d0} {d1} {d2} --out {out}

## Notes
- **fetch_push is hard:** watch `final_dist` / `min_dist` (object -> target) — they
  fall before success rises. If `final_dist` is still dropping at 500k, raise `STEPS`.
- **Disconnected?** Re-run cells 1-4 with the same `SEED`, set `RESUME = True`, and
  run cell 5 to continue from `latest.pkl`.
- **Checkpoints:** `latest.pkl` (resume), `best.pkl` (best eval), `metrics.json`.
- Existing FetchReach seed-0 run may be named `fetch_reach_nce`; the multi-seed
  cell expects `fetch_reach_nce_s0` — rerun seed 0 with this notebook, or edit `d0`.